# Metalearner PEHE Benchmark

Compares S-Learner, T-Learner, X-Learner, and DR-Learner across three synthetic DGPs
using PEHE (Precision in Estimating Heterogeneous Effects) as the scalar evaluation metric.

**DGP 1** — linear outcome, constant treatment effect (τ = 2), no confounding (e = 0.5).

**DGP 2** — nonlinear outcome (sigmoid), heterogeneous CATE (τ = 2·X₀), moderate
confounding (propensity correlated with X₀).

**DGP 3** — same as DGP 2 but with strong confounding; limited propensity overlap.

The benchmark quantifies how much confounding hurts each learner and whether
doubly-robust / cross-fitted corrections help recover accuracy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from econml.metalearners import SLearner, TLearner, XLearner
from econml.dr import DRLearner
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor

%matplotlib inline

In [ ]:
N, D = 2000, 5


def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def dgp1(seed=42, n=N, d=D):
    """Linear outcome, constant tau=2, balanced assignment (no confounding)."""
    rng = np.random.RandomState(seed)
    X = rng.randn(n, d)
    beta = rng.randn(d)
    tau_true = np.full(n, 2.0)
    prop = np.full(n, 0.5)
    T = rng.binomial(1, prop)
    Y = X @ beta + tau_true * T + 0.5 * rng.randn(n)
    return X, T, Y, tau_true


def dgp2(seed=42, n=N, d=D, conf_strength=0.5):
    """Sigmoid outcome, heterogeneous tau=2*X0, confounding via logistic propensity."""
    rng = np.random.RandomState(seed)
    X = rng.randn(n, d)
    beta = rng.randn(d)
    tau_true = 2.0 * X[:, 0]
    prop = _sigmoid(conf_strength * X[:, 0])
    T = rng.binomial(1, prop)
    Y = _sigmoid(X @ beta) + tau_true * T + 0.3 * rng.randn(n)
    return X, T, Y, tau_true


def dgp3(seed=42, n=N, d=D):
    """Same as DGP2 but strong confounding (conf_strength=2.0)."""
    return dgp2(seed=seed, n=n, d=d, conf_strength=2.0)

In [ ]:
DGPS = {
    "DGP1 (linear, no confounding)": dgp1,
    "DGP2 (nonlinear, moderate confounding)": dgp2,
    "DGP3 (nonlinear, strong confounding)": dgp3,
}

LEARNER_NAMES = ["S-Learner", "T-Learner", "X-Learner", "DR-Learner"]


def pehe(tau_hat, tau_true):
    return float(np.sqrt(np.mean((tau_hat - tau_true) ** 2)))


results = {}

for dgp_label, dgp_fn in DGPS.items():
    X, T, Y, tau_true = dgp_fn(seed=42)
    row = {}

    # S-Learner: single model trained on (X, T) concatenated
    sl = SLearner(overall_model=LinearRegression())
    sl.fit(Y, T, X=X)
    row["S-Learner"] = pehe(sl.effect(X).flatten(), tau_true)

    # T-Learner: separate models per treatment arm
    tl = TLearner(models=LinearRegression())
    tl.fit(Y, T, X=X)
    row["T-Learner"] = pehe(tl.effect(X).flatten(), tau_true)

    # X-Learner: propensity-weighted correction on top of T-Learner residuals
    xl = XLearner(
        models=LinearRegression(),
        cate_models=RandomForestRegressor(n_estimators=100, random_state=42),
    )
    xl.fit(Y, T, X=X)
    row["X-Learner"] = pehe(xl.effect(X).flatten(), tau_true)

    # DR-Learner: doubly-robust pseudo-outcome regression with cross-fitting
    dr = DRLearner(
        model_regression=LinearRegression(),
        model_propensity=LogisticRegression(max_iter=500),
        model_final=LinearRegression(),
        random_state=42,
    )
    dr.fit(Y, T, X=X)
    row["DR-Learner"] = pehe(dr.effect(X).flatten(), tau_true)

    results[dgp_label] = row

# Print summary table
col_w = 13
header = f"{'DGP':45s}" + "".join(f"{n:>{col_w}s}" for n in LEARNER_NAMES)
print(header)
print("-" * len(header))
for label, row in results.items():
    print(f"{label:45s}" + "".join(f"{row[n]:{col_w}.4f}" for n in LEARNER_NAMES))

In [ ]:
pehe_matrix = np.array([[results[d][l] for l in LEARNER_NAMES] for d in DGPS])
dgp_labels = list(DGPS.keys())

fig, ax = plt.subplots(figsize=(9, 3.5))
im = ax.imshow(pehe_matrix, cmap="YlOrRd", aspect="auto")

ax.set_xticks(range(len(LEARNER_NAMES)))
ax.set_xticklabels(LEARNER_NAMES, fontsize=12)
ax.set_yticks(range(len(dgp_labels)))
ax.set_yticklabels(dgp_labels, fontsize=10)

thresh = 0.65 * pehe_matrix.max()
for i in range(pehe_matrix.shape[0]):
    for j in range(pehe_matrix.shape[1]):
        color = "white" if pehe_matrix[i, j] > thresh else "black"
        ax.text(
            j, i, f"{pehe_matrix[i, j]:.3f}",
            ha="center", va="center", fontsize=11, color=color,
        )

plt.colorbar(im, ax=ax, label="PEHE (lower is better)")
ax.set_title("PEHE by Metalearner and DGP", fontsize=13, pad=10)
plt.tight_layout()
plt.show()